 # 5 工具m

前置：加载工具

In [2]:
from dotenv import load_dotenv
import os

# 关键：每次运行都强制覆盖系统环境变量
load_dotenv(override=True)

# 然后读取
api_key = os.getenv('DEEPSEEK_API_KEY')

In [21]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)

print(response.choices[0].message.content)

Hello! How can I help you today?


5.1初体验，模拟获取天气，了解工具的基本组成

In [ ]:
# 1.使用tool装饰器定义工具
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

5.1.2初体验，创建智能体，并绑定工具（用数组的方式绑定）

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 2.创建智能体，并绑定工具
agent = create_agent(
    model="	deepseek-v4-flash",
    tools=[get_weather]
)

# 3.调用Agent
response = agent.invoke(
    {"messages": [HumanMessage(content="杭州今天天气如何?")]},
)

for message in response['messages']:
    message.pretty_print()

5.2定义工具的几种形式

5.2.1智能体在工作时，需要将函数的名称、输入、作用传递给大模型，默认情况下这些信息的来源是：
- 工具名称：函数名
- 工具输入：函数入参
- 工具作用：函数的注释

In [ ]:
# 定义工具
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """计算指定数字的平方根"""
    return x ** 0.5

5.2.2当然，我们可以通过tool装饰器来覆盖上述信息：
- 通过装饰器定义工具名称

In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

5.2.2- 通过装饰器定义工具作用描述

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

5.2.3- 通过装饰器定义工具入参约束
如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束。

例如，我们需要定义个查询天气的tool，借助于Pydantic来约束入参。
我们定义一个入参的模型，在模型中添加入参描述信息：

In [ ]:
# 例如一个查询天气的tool
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

5.2.4定义工具，使用定义的模型来约束入参：

In [ ]:
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

工具定义好之后，调用方式与普通函数类似：

In [ ]:
# 调用数学工具
tool1.invoke({"x": 467})

# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})

当我们创建智能体时，可以把定义好的工具传递给智能体，将来模型就能得到工具信息，并根据情况判断是否需要调用工具，需要调用哪个工具了。m

In [ ]:
from langchain.agents import create_agent

# 创建智能体，并添加工具
agent = create_agent(
    model="deepseek-chat",
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

接下来，调用智能体，向其提问，模型会自动根据用户问题判断：
- 是否需要调用工具？
- 该调用哪个工具？
- 该传递那些参数？
并且在调用工具之后，根据工具执行结果给用户生成响应。

In [ ]:
# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

如果采用stream模式的updates模式，可以看到工具调用的具体步骤

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

# 5.3预定义工具m

LangChain中提供了很多预定义好的工具，方便我们使用，可使用的预定义工具列表可参考官网：
https://www.langchain.com.cn/docs/integrations/tools/m

5.3.1使用工具m

In [22]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

# 初始化工具，并设置参数，具体参数设置参考官网
tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

 完整的参数设置参考官网
https://docs.langchain.com/oss/python/integrations/tools/tavily_search
https://docs.tavily.com/documentation/api-reference/endpoint/search

In [23]:
tool.invoke("杭州今天天气如何？")

{'query': '杭州今天天气如何？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E6%B5%99%E6%B1%9F%E6%9D%AD%E5%B7%9E%E5%A4%A9%E6%B0%94&srcid=4982',
   'title': '杭州 - 百度',
   'content': '今天. 14 ~ 23°C 多云. 东风2级. 优 · 09. 16 ~ 25°C 多云. 东南风2级. 良 · 10. 19 ~ 28°C 多云. 东南风2级. 良.',
   'score': 0.9997241,
   'raw_content': None},
  {'url': 'https://www.accuweather.com/zh/cn/hangzhou/106832/weather-forecast/106832',
   'title': '杭州, 浙江省, 中國三日天氣預報 - AccuWeather',
   'content': '# 杭州, 浙江省. 今天 WinterCast 當地{stormName}追蹤 每小時 十天賽 雷達 MinuteCast® 每月 空氣品質 健康與活動. ### 颶風 ### 惡劣天氣 ### 雷達與氣象圖 ### 視訊. ## 今天  每小時   十天賽   雷達   MinuteCast®   每月   空氣品質   健康與活動. ## 目前天氣. ## 杭州 氣象雷達圖. Thank you for your patience as we work to get everything up and running again. ## 每小時預測. rain drop 0%  上午4时   15°. rain drop 0%  上午5时   15°. rain drop 0%  上午6时   15°. rain drop 0%  上午7时   18°. rain drop 0%  上午8时   20°. rain drop 0%  上午9时   23°. rain drop 0%

5.3.5结合智能体

In [24]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

# 调用工具
for chunk in agent.stream(
    {"messages": [HumanMessage(content="北京接下来5天天气如何?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

step: model
content: [{'type': 'text', 'text': '我来帮你查一下北京接下来5天的天气情况。'}, {'type': 'tool_call', 'id': 'call_00_qhPf8HvDyvelVQGcR2Sn7207', 'name': 'tavily_search', 'args': {'query': '北京未来5天天气预报 2025', 'search_depth': 'advanced'}}]

step: tools
content: [{'type': 'text', 'text': '{"query": "北京未来5天天气预报 2025", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.ysxiao.cn/beijingyoushengxiao/212198.html", "title": "2025年5月12日-5月19日期间北京市天气预报 - 北京幼升小网", "content": "以下是2025年5月12日至5月19日期间北京市的天气预报情况：\\n\\n| 日期 | 日间天气现象 | 日间最高气温（℃） | 日间最低气温（℃） | 日间风力等级 | 日间风向 | 夜间天气现象 | 夜间风力等级 | 夜间风向 |\\n ---  ---  ---  --- \\n| 2025年5月12日 | 多云 | 27 | 17 | 1级 | 南风 | 晴 | 1级 | 南风 |\\n| 2025年5月13日 | 晴 | 27 | 19 | 1级 | 东南风 | 多云 | 1级 | 东北风 |\\n| 2025年5月14日 | 雷阵雨 | 25 | 16 | 1级 | 东北风 | 多云 | 1级 | 西南风 |\\n| 2025年5月15日 | 晴 | 27 | 17 | 3级 | 西南风 | 晴 | 1级 | 西南风 |\\n| 2025年5月16日 | 多云 | 28 | 19 | 1级 | 南风 | 多云 | 1级 | 南风 |\\n| 2025年5月17日 | 阴 | 29 | 17 | 1级 | 东北风 | 阴 | 1级 | 东风 |\\n| 2025年5月18

5.3.6.优化
注意，LangChain提供的TavilySearch工具描述非常复杂，参数也很多。会有额外的网络消耗。如果我们仅仅是需要query参数，建议自定义工具。m

In [3]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

# 使用tavily作为web搜索工具
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

5.3.6自定义结构化输出

In [32]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response)

{'messages': [HumanMessage(content='蒸蚌是什么梗？', additional_kwargs={}, response_metadata={}, id='2729c2ce-3b36-4425-8bc7-849def875488'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 439, 'total_tokens': 480, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 55}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '3b473dba-7e06-4865-828e-cd5e09578258', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e075a-b52e-7662-99b1-11fcdcdea9c6-0', tool_calls=[{'name': 'web_search', 'args': {'query': '蒸蚌 梗 是什么'}, 'id': 'call_00_lv2KEHfiakijK0FvTeqt2421', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 439, 'output_tokens': 41, 'total_tokens': 480, 'input_token_deta